# RSNA Knee Abnormality Detection — standalone submission notebook

**Pipeline (identical to EXP-004 / EXP-008 / EXP-011):** frozen DINOv2-Base CLS embeddings of 9 slices per study, mean-pooled, then a per-target `StandardScaler + LogisticRegression(C=0.01, class_weight="balanced")` trained on the 58 fully-labelled training studies. The regularisation setting `C=0.01` is the current candidate, selected after the OOF comparison documented below; it is not yet proven to improve leaderboard score. Test reports are never used.

**What is different from the experiment notebook**
- Self-contained: nothing is read from `/kaggle/working`, GitHub, or the internet. Training features are recomputed here (about 30 s).
- Never crashes on a bad study: series fall back sagittal-fluid → coronal-fluid → any sagittal → any coronal → axial-fluid → any series; slice ordering and sampling tolerate missing/duplicate `InstanceNumber` and short series; unreadable studies get the training prior.
- A valid prior-only `submission.csv` is written first, so a file exists before inference. Any pre-existing submission is backed up before replacement. The final file is validated; invalid model output leaves the prior-only file in place.
- Everything is logged to `run_report.json`.

**Notebook settings:** Internet **OFF**, Accelerator **GPU ON**, inputs = the competition + the DINOv2 model (`metaresearch/dinov2`, PyTorch, base).

**Before you submit:**
1. Set `SIMULATE_N = 100`, run the notebook, and read the report: fallback rates and seconds per study.
2. Set `SIMULATE_N = 0`.
3. **Save Version → Save & Run All**, open the version's output, and check `run_report.json` says `status: OK`.
4. Only then submit from the version page.

## 1. Setup and configuration

Run the setup cells below in order. They define paths, settings, DICOM loading, feature extraction, and the prediction pipeline. If the Kaggle session restarts, rerun these cells before running any later cell.

**Normal submission run:** execute the setup cells, then go to **Generate the submission**. The validation experiments are optional and are not needed to create `submission.csv`.

In [1]:
import gc, json, os, shutil, sys, time, traceback, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------ settings ------------------------------
DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"
WORK = Path("/kaggle/working")
SUBMISSION_PATH = WORK / "submission.csv"

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_SLICES = 9
# (0.0, 1.0) reproduces EXP-004/008/011 exactly (first and last slice included).
# Example for central slices only: (0.15, 0.85). Training and test use the same setting.
SLICE_RANGE = (0.0, 1.0)
SEED = 42

SIMULATE_N = 0                    # >0 = dry run on this many unlabelled TRAIN studies; MUST be 0 to submit
TIME_BUDGET_SECONDS = 7.5 * 3600  # stop extracting and use the prior for the rest (competition limit is 9 h)
MAX_SERIES_TRIES = 4              # series attempted per study before giving up on it
MIN_TRAIN_STUDIES = 50            # fewer extracted gold studies than this -> try an attached feature cache
MIN_MODEL_STUDIES = 20            # fewer than this -> prior-only submission
HIGH_FAILURE_RATE = 0.20          # status becomes WARN if more test studies than this fall back to the prior

warnings.filterwarnings("ignore")
START = time.time()
LOG = []   # one dict per fallback / failure event; summarised in run_report.json

print("pydicom", pydicom.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("SIMULATE_N =", SIMULATE_N, "(must be 0 for the real submission)" if SIMULATE_N else "(real submission mode)")

pydicom 3.0.2 | numpy 2.1.3 | pandas 2.3.3
SIMULATE_N = 0 (real submission mode)


In [2]:
def log(event, **kw):
    LOG.append({"event": event, "t": round(time.time() - START, 1), **kw})


def elapsed():
    return time.time() - START


# ------------------------------ model ------------------------------
def find_model_dir():
    p = Path(MODEL_PATH)
    if (p / "config.json").exists():
        return str(p)
    # Fall back to any attached DINOv2 model (only under /kaggle/input/models, never the DICOM folders).
    for cfg in sorted(Path("/kaggle/input/models").glob("**/config.json")):
        try:
            if "dinov2" in cfg.read_text().lower():
                log("model_path_fallback", path=str(cfg.parent))
                return str(cfg.parent)
        except Exception:
            pass
    raise FileNotFoundError(f"DINOv2 weights not found at {MODEL_PATH} or under /kaggle/input/models")


def load_encoder():
    """Same extraction as EXP-004: CLS token of last_hidden_state, 768-d per slice."""
    import torch
    from transformers import AutoImageProcessor, AutoModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    path = find_model_dir()
    processor = AutoImageProcessor.from_pretrained(path, local_files_only=True)
    model = AutoModel.from_pretrained(path, local_files_only=True).to(device).eval()

    def encode(images):
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model(**inputs)
        return out.last_hidden_state[:, 0, :].float().cpu().numpy()

    return encode, str(device)


# ------------------------------ series choice ------------------------------
def ranked_series(series_df, study_id):
    """Ordered list of (series_id, rung). The first entry matches the EXP-004 choice when a fluid sagittal exists."""
    c = series_df[series_df["StudyInstanceUID"].astype(str) == str(study_id)]
    if c.empty:
        return []
    plane = (c["Anatomical_Plane"].astype(str).str.strip().str.lower()
             if "Anatomical_Plane" in c else pd.Series("", index=c.index))
    fluid = ((pd.to_numeric(c["Fluid_Sensitive"], errors="coerce") == 1)
             if "Fluid_Sensitive" in c else pd.Series(False, index=c.index))
    everything = pd.Series(True, index=c.index)
    rungs = [
        ("sagittal_fluid", (plane == "sagittal") & fluid),
        ("coronal_fluid", (plane == "coronal") & fluid),
        ("sagittal_any", plane == "sagittal"),
        ("coronal_any", plane == "coronal"),
        ("axial_fluid", (plane == "axial") & fluid),
        ("any_series", everything),
    ]
    out, seen = [], set()
    for name, mask in rungs:
        for sid in sorted(c.loc[mask, "SeriesInstanceUID"].astype(str)):
            if sid not in seen:
                seen.add(sid)
                out.append((sid, name))
    return out


# ------------------------------ DICOM ------------------------------
def ordered_dicom_paths(folder):
    """Sort slices by InstanceNumber; fall back to position along the slice normal, then to file name."""
    paths = sorted(Path(folder).glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM files in {folder}")
    heads = []
    for p in paths:
        try:
            heads.append((p, pydicom.dcmread(p, stop_before_pixels=True)))
        except Exception:
            log("header_unreadable", file=str(p))
    if not heads:
        raise ValueError(f"No readable DICOM headers in {folder}")

    inst = [getattr(ds, "InstanceNumber", None) for _, ds in heads]
    if all(i is not None for i in inst) and len({int(i) for i in inst}) == len(inst):
        order = sorted(range(len(heads)), key=lambda k: (int(inst[k]), heads[k][0].name))
        return [heads[k][0] for k in order], "instance_number"
    try:
        iop = np.array(heads[0][1].ImageOrientationPatient, dtype=float)
        normal = np.cross(iop[:3], iop[3:])
        pos = [float(np.dot(normal, np.array(ds.ImagePositionPatient, dtype=float))) for _, ds in heads]
        order = sorted(range(len(heads)), key=lambda k: (pos[k], heads[k][0].name))
        return [heads[k][0] for k in order], "image_position"
    except Exception:
        return sorted(h[0] for h in heads), "filename"


def dicom_to_image(path):
    """Identical preprocessing to EXP-004."""
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2:
        raise ValueError(f"Unsupported pixel array shape {pixels.shape}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1))
    pixels += float(getattr(ds, "RescaleIntercept", 0))
    lo, hi = np.percentile(pixels, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        raise ValueError(f"Invalid intensity range: {path}")
    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
    pixels = (pixels * 255).astype(np.uint8)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = 255 - pixels
    return Image.fromarray(pixels).convert("RGB")


def sample_indices(n):
    lo, hi = SLICE_RANGE
    return np.rint(np.linspace(lo * (n - 1), hi * (n - 1), N_SLICES)).astype(int)


def extract_study(encode, series_df, root, study_id):
    """Return ((9, 768) features, info). Tries up to MAX_SERIES_TRIES series; raises only if all fail."""
    tries = ranked_series(series_df, study_id)[:MAX_SERIES_TRIES]
    if not tries:
        raise RuntimeError("study has no rows in the series table")
    last_err = None
    for rank, (sid, rung) in enumerate(tries):
        try:
            paths, order_mode = ordered_dicom_paths(Path(root) / str(study_id) / sid)
            idx = sample_indices(len(paths)).tolist()
            images, failed = {}, 0
            for k in sorted(set(idx)):
                try:
                    images[k] = dicom_to_image(paths[k])
                except Exception as e:
                    failed += 1
                    log("slice_failed", study=str(study_id), series=sid, slice=k, error=repr(e)[:160])
            if not images:
                raise RuntimeError("no readable sampled slices")
            readable = sorted(images)
            # An unreadable slice is replaced by its nearest readable neighbour.
            batch = [images[min(readable, key=lambda j: abs(j - k))] for k in idx]
            feats = np.asarray(encode(batch), dtype=np.float32)
            if feats.shape != (N_SLICES, 768) or not np.isfinite(feats).all():
                raise RuntimeError(f"invalid features {feats.shape}")
            info = {"series_id": sid, "rung": rung, "rung_rank": rank, "order_mode": order_mode,
                    "n_slices": len(paths), "unique_slices_sampled": len(set(idx)),
                    "unreadable_slices": failed}
            return feats, info
        except Exception as e:
            last_err = e
            log("series_failed", study=str(study_id), series=sid, rung=rung, error=repr(e)[:200])
    raise RuntimeError(f"all {len(tries)} candidate series failed; last error: {last_err!r}")

In [3]:
def resolve_inputs():
    """Work out which studies to predict, which columns to write, and where their DICOMs live."""
    if SIMULATE_N > 0:
        labels = pd.read_csv(DATA / "train.csv")
        gold_ids = set(labels.dropna(subset=TARGETS)["StudyInstanceUID"].astype(str))
        pool = sorted(s for s in labels["StudyInstanceUID"].astype(str) if s not in gold_ids)
        return dict(ids=pool[:SIMULATE_N], id_col="StudyInstanceUID",
                    columns=["StudyInstanceUID"] + TARGETS, root=DATA / "train_series",
                    series=pd.read_csv(DATA / "train_series.csv"),
                    out=WORK / "simulated_submission.csv", simulated=True)

    test_ids = pd.read_csv(DATA / "test.csv")["StudyInstanceUID"].astype(str).tolist()
    if not test_ids:
        raise ValueError("test.csv contains no study IDs")
    if len(set(test_ids)) != len(test_ids):
        raise ValueError("test.csv contains duplicate StudyInstanceUID values")

    # The official test.csv is authoritative for IDs and row order. The sample
    # submission may define column order, but must never replace the test IDs.
    columns = ["StudyInstanceUID"] + TARGETS
    sample_path = DATA / "sample_submission.csv"
    if sample_path.exists():
        sample = pd.read_csv(sample_path)
        if len(sample.columns) > 1:
            sample_columns = sample.columns.tolist()
            if sample_columns[0] != "StudyInstanceUID" or set(sample_columns[1:]) != set(TARGETS) or len(sample_columns) != len(TARGETS) + 1:
                raise ValueError(f"Unexpected sample_submission schema: {sample_columns}")
            columns = sample_columns
            sample_ids = sample["StudyInstanceUID"].astype(str).tolist()
            if sample_ids != test_ids:
                log("sample_vs_test_id_mismatch", n_sample=len(sample_ids), n_test=len(test_ids),
                    action="using test.csv IDs and order")

    return dict(ids=test_ids, id_col="StudyInstanceUID", columns=columns, root=DATA / "test_series",
                series=pd.read_csv(DATA / "test_series.csv"), out=SUBMISSION_PATH, simulated=False)


def prior_table(inputs, prior):
    table = pd.DataFrame({inputs["id_col"]: inputs["ids"]})
    for col in inputs["columns"][1:]:
        table[col] = float(prior.get(col, 0.5))
    return table[inputs["columns"]]


def build_training_set(encode, state):
    """Recompute features for the fully-labelled studies; fall back to an attached cache if that fails."""
    labels = pd.read_csv(DATA / "train.csv")
    series = pd.read_csv(DATA / "train_series.csv")
    gold = labels.dropna(subset=TARGETS).sort_values("StudyInstanceUID").reset_index(drop=True)
    if len(gold) != 58:
        log("unexpected_gold_count", n=len(gold))
    feats, kept = [], []
    for _, row in gold.iterrows():
        sid = str(row["StudyInstanceUID"])
        try:
            f, info = extract_study(encode, series, DATA / "train_series", sid)
            feats.append(f.mean(axis=0))
            kept.append(sid)
            state["train_rungs"][info["rung"]] += 1
        except Exception as e:
            log("train_study_failed", study=sid, error=repr(e)[:200])
    print(f"Training features: {len(kept)}/{len(gold)} gold studies in {elapsed():.0f}s")
    X, gold_kept = (np.stack(feats) if feats else np.zeros((0, 768), np.float32)), gold[gold["StudyInstanceUID"].astype(str).isin(kept)]

    if len(kept) < MIN_TRAIN_STUDIES:
        # Look for an attached copy of the EXP-004 cache (features.npy + study_ids.npy + labels.csv).
        for feat_path in sorted(Path("/kaggle/input").glob("*/features.npy")) + sorted(Path("/kaggle/input").glob("*/*/features.npy")):
            d = feat_path.parent
            try:
                f3 = np.load(d / "features.npy")
                ids = np.load(d / "study_ids.npy", allow_pickle=True).astype(str)
                lab = pd.read_csv(d / "labels.csv")
                if f3.ndim == 3 and f3.shape[1:] == (N_SLICES, 768) and len(ids) == len(f3) == len(lab):
                    lab = lab.assign(StudyInstanceUID=lab["StudyInstanceUID"].astype(str))
                    if lab["StudyInstanceUID"].tolist() == ids.tolist() and lab[TARGETS].notna().all().all():
                        log("train_cache_fallback", path=str(d), n=len(ids))
                        state["notes"].append(f"training features loaded from attached cache {d}")
                        return f3.mean(axis=1).astype(np.float32), lab
            except Exception as e:
                log("train_cache_unusable", path=str(d), error=repr(e)[:160])
    return X, gold_kept.reset_index(drop=True)


def fit_models(X, gold):
    models = {}
    for t in TARGETS:
        y = pd.to_numeric(gold[t], errors="coerce").to_numpy()
        ok = np.isfinite(y)
        if ok.sum() < 10 or len(np.unique(y[ok])) < 2:
            models[t] = None
            log("target_without_model", target=t, n=int(ok.sum()))
            continue
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(C=0.01, class_weight="balanced", max_iter=2000, random_state=SEED))
        clf.fit(X[ok], y[ok].astype(int))
        models[t] = clf
    return models


def validate_submission(sub, columns, ids):
    problems = []
    if sub.columns.tolist() != columns:
        problems.append("columns differ from the expected list")
    if len(sub) != len(ids):
        problems.append(f"row count {len(sub)} != {len(ids)}")
    elif sub[columns[0]].astype(str).tolist() != list(ids):
        problems.append("study id order differs")
    vals = sub[columns[1:]].to_numpy(dtype=np.float64)
    if not np.isfinite(vals).all():
        problems.append("NaN or infinite predictions")
    elif (vals < 0).any() or (vals > 1).any():
        problems.append("predictions outside [0, 1]")
    return problems


def run_pipeline():
    """Never raises. A prior-only submission is written first and only replaced by a validated model submission."""
    state = {"status": "OK", "notes": [], "device": None, "train_rungs": Counter(), "test_rungs": Counter(),
             "order_modes": Counter(), "n_test": 0, "n_model": 0, "n_prior_filled": 0, "seconds_per_study": None}
    prior, inputs = {}, None
    try:
        inputs = resolve_inputs()
        state["n_test"] = len(inputs["ids"])
        state["simulated"] = inputs["simulated"]
        try:
            lab = pd.read_csv(DATA / "train.csv").dropna(subset=TARGETS)
            prior = {t: float(pd.to_numeric(lab[t]).mean()) for t in TARGETS}
        except Exception as e:
            log("prior_unavailable", error=repr(e)[:200])
        # Preserve any pre-existing submission before creating the safety-net file.
        if inputs["out"].exists():
            backup = WORK / f"submission.csv.backup.{time.strftime('%Y%m%d_%H%M%S')}"
            shutil.copy2(inputs["out"], backup)
            log("existing_submission_backed_up", path=str(backup))
        prior_table(inputs, prior).to_csv(inputs["out"], index=False)
        print(f"[step 0] prior-only safety-net file written: {inputs['out']} ({len(inputs['ids'])} studies)")

        encode, state["device"] = load_encoder()
        print("[step 1] encoder loaded on", state["device"])
        if state["device"] == "cpu":
            state["notes"].append("running on CPU; extraction will be slow")
        X_train, gold = build_training_set(encode, state)
        if len(X_train) < MIN_MODEL_STUDIES:
            raise RuntimeError(f"only {len(X_train)} usable training studies; keeping prior-only submission")
        models = fit_models(X_train, gold)
        print(f"[step 2] fitted {sum(m is not None for m in models.values())}/{len(TARGETS)} target models on {len(X_train)} studies")

        feats, ok_ids, cache = [], [], {}
        t0 = time.time()
        for i, sid in enumerate(inputs["ids"], start=1):
            if elapsed() > TIME_BUDGET_SECONDS:
                log("time_budget_exceeded", processed=i - 1, remaining=len(inputs["ids"]) - i + 1)
                state["notes"].append("time budget exceeded; remaining studies use the prior")
                break
            if sid not in cache:
                try:
                    f, info = extract_study(encode, inputs["series"], inputs["root"], sid)
                    cache[sid] = f.mean(axis=0)
                    state["test_rungs"][info["rung"]] += 1
                    state["order_modes"][info["order_mode"]] += 1
                except Exception as e:
                    cache[sid] = None
                    log("study_failed", study=sid, error=repr(e)[:200])
            if i % 25 == 0 or i == len(inputs["ids"]):
                print(f"[step 3] {i}/{len(inputs['ids'])} studies | {time.time() - t0:.0f}s")
        done = max(1, sum(v is not None for v in cache.values()) + sum(v is None for v in cache.values()))
        state["seconds_per_study"] = round((time.time() - t0) / done, 3)

        sub = prior_table(inputs, prior)
        have = [k for k, sid in enumerate(inputs["ids"]) if cache.get(sid) is not None]
        state["n_model"] = len(have)
        state["n_prior_filled"] = len(inputs["ids"]) - len(have)
        if have:
            Xt = np.stack([cache[inputs["ids"][k]] for k in have])
            for t in TARGETS:
                if t in sub.columns and models[t] is not None:
                    sub.loc[have, t] = models[t].predict_proba(Xt)[:, 1]

        problems = validate_submission(sub, inputs["columns"], inputs["ids"])
        if problems:
            state["status"] = "INVALID_USING_PRIOR_ONLY"
            state["notes"] += problems
            print("[step 4] VALIDATION FAILED, keeping the prior-only file:", problems)
        else:
            # Validate a temporary file first; replace the safety-net file only
            # after a successful write and read-back validation.
            tmp_out = inputs["out"].with_name(inputs["out"].name + ".tmp")
            try:
                sub.to_csv(tmp_out, index=False)
                back = pd.read_csv(tmp_out)
                problems = validate_submission(back, inputs["columns"], inputs["ids"])
                if problems:
                    state["status"] = "INVALID_READBACK"
                    state["notes"] += problems
                    tmp_out.unlink(missing_ok=True)
                else:
                    os.replace(tmp_out, inputs["out"])
            except Exception:
                tmp_out.unlink(missing_ok=True)
                raise
            if not problems:
                rate = state["n_prior_filled"] / max(1, state["n_test"])
                if state["n_model"] == 0:
                    state["status"] = "PRIOR_ONLY"
                elif rate > HIGH_FAILURE_RATE:
                    state["status"] = "WARN_HIGH_FAILURE_RATE"
                elif any(e["event"] == "time_budget_exceeded" for e in LOG):
                    state["status"] = "WARN_TIME_BUDGET_EXCEEDED"
                print(f"[step 4] wrote and re-validated {inputs['out']} | model predictions for {state['n_model']}/{state['n_test']} studies")
        state["prediction_summary"] = {
            t: {"mean": round(float(sub[t].mean()), 4), "std": round(float(sub[t].std(ddof=0)), 4)}
            for t in TARGETS if t in sub.columns}
    except Exception:
        state["status"] = "ERROR_PRIOR_ONLY" if inputs is not None else "ERROR_NO_FILE"
        state["notes"].append(traceback.format_exc()[-1500:])
        print("PIPELINE ERROR (prior-only file, if any, left in place):\n", traceback.format_exc())
    finally:
        gc.collect()
    return state

In [4]:
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

## 2. Generate the submission

**This is the main workflow.** Run the next cell to generate predictions and write `submission.csv`, then run the report cell to inspect the run status and fallback counts.

Before running, check the settings cell: `SIMULATE_N` must be `0` for a real competition submission. A positive value is only for a dry run and must not be submitted.

In [ ]:
RESULT = run_pipeline()
print("\nstatus:", RESULT["status"])

In [ ]:
report = {
    "status": RESULT["status"],
    "simulated_run": bool(RESULT.get("simulated")),
    "n_studies": RESULT["n_test"],
    "n_with_model_predictions": RESULT["n_model"],
    "n_filled_with_prior": RESULT["n_prior_filled"],
    "seconds_per_study_extraction": RESULT["seconds_per_study"],
    "total_elapsed_seconds": round(elapsed(), 1),
    "device": RESULT["device"],
    "series_rung_used_train": dict(RESULT["train_rungs"]),
    "series_rung_used_test": dict(RESULT["test_rungs"]),
    "slice_order_modes_test": dict(RESULT["order_modes"]),
    "slice_range": list(SLICE_RANGE),
    "notes": RESULT["notes"],
    "prediction_summary": RESULT.get("prediction_summary", {}),
    "event_counts": dict(Counter(e["event"] for e in LOG)),
    "first_events": LOG[:60],
}
try:
    (WORK / "run_report.json").write_text(json.dumps(report, indent=2, default=str))
except Exception as e:
    print("could not write run_report.json:", e)

print("=" * 70)
print("STATUS:", report["status"])
print(f"studies: {report['n_studies']} | model predictions: {report['n_with_model_predictions']} | prior-filled: {report['n_filled_with_prior']}")
print("rungs (test):", report["series_rung_used_test"], "| order modes:", report["slice_order_modes_test"])
print("events:", report["event_counts"])
print("seconds per study:", report["seconds_per_study_extraction"], "| total seconds:", report["total_elapsed_seconds"])
if report["simulated_run"]:
    print("\nSIMULATED RUN: no real submission was written. Set SIMULATE_N = 0 before the real run.")
elif report["status"] != "OK":
    print("\n>>> DO NOT SUBMIT YET: status is not OK. Read run_report.json notes and events above. <<<")
else:
    print("\nOK: submission.csv written and validated.")
print("=" * 70)
if report["notes"]:
    print("notes:", report["notes"])

## 3. Submission audit (optional)

This checks an already-generated real `submission.csv` against the competition test schema. It does not retrain the model or change predictions. Run it only after a successful real submission run.


## Audit Existing Real Submission (CPU Only)

Validate the existing `submission.csv` against the competition's
`test.csv` and expected submission schema.

Checks:
- Exact columns and column order
- Full test row count
- Exact study ID order
- Finite prediction values within [0, 1]
- SHA-256 checksum for artifact identification

This cell does not rerun inference or modify predictions.
It saves `real_submission_audit.json` separately from the
simulation's `run_report.json`.

**Expected result:** `status: PASS`.


In [9]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
WORK = Path("/kaggle/working")
SUB = WORK / "submission.csv"

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

assert SUB.exists(), f"Missing real submission: {SUB}"

test = pd.read_csv(DATA / "test.csv")
sub = pd.read_csv(SUB)
sample_path = DATA / "sample_submission.csv"
expected_cols = (
    pd.read_csv(sample_path, nrows=0).columns.tolist()
    if sample_path.exists()
    else ["StudyInstanceUID"] + TARGETS
)

# Structural checks
assert sub.columns.tolist() == expected_cols, (
    f"Column mismatch: got {sub.columns.tolist()}, "
    f"expected {expected_cols}"
)
assert len(sub) == len(test), (
    f"Wrong row count: submission={len(sub)}, test={len(test)}"
)
assert sub["StudyInstanceUID"].astype(str).tolist() == (
    test["StudyInstanceUID"].astype(str).tolist()
), "Submission study IDs/order do not match test.csv"

values = sub[TARGETS].to_numpy(dtype=float)
assert np.isfinite(values).all(), "NaN or infinite predictions found"
assert ((values >= 0) & (values <= 1)).all(), (
    "Predictions outside [0, 1]"
)

sha256 = hashlib.sha256(SUB.read_bytes()).hexdigest()

# Record the artifact validation separately from the simulation report.
audit = {
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "file": str(SUB),
    "sha256": sha256,
    "n_rows": len(sub),
    "n_expected_test_rows": len(test),
    "columns": sub.columns.tolist(),
    "ids_and_order_match_test_csv": True,
    "predictions_finite": True,
    "predictions_within_0_1": True,
    "status": "PASS",
    "provenance_note": (
        "Artifact validation only. The earlier real-run report was "
        "overwritten by the later simulation; this does not reconstruct "
        "the earlier run's runtime or event log."
    ),
}

(WORK / "real_submission_audit.json").write_text(
    json.dumps(audit, indent=2)
)

print(json.dumps(audit, indent=2))


{
  "validated_at_utc": "2026-10-15T20:23:26.368940+00:00",
  "file": "/kaggle/working/submission.csv",
  "sha256": "0897242c9f931a1f12f7f597288b6243ebe4cf3ede9d8430145f3ccba66c5ab7",
  "n_rows": 3,
  "n_expected_test_rows": 3,
  "columns": [
    "StudyInstanceUID",
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
  ],
  "ids_and_order_match_test_csv": true,
  "predictions_finite": true,
  "predictions_within_0_1": true,
  "status": "PASS",
  "provenance_note": "Artifact validation only. The earlier real-run report was overwritten by the later simulation; this does not reconstruct the earlier run's runtime or event log."
}


## 4. Model validation experiments (optional; do not run for a normal submission)

These cells compare regularisation settings using out-of-fold (OOF) predictions on the 58 fully labelled studies. Cell 15 is the most complete comparison: it rebuilds the labelled feature matrix and compares `C=1.0`, `C=0.01`, and `C=0.03` using identical folds for each target and seed.

### Findings (run recorded in this notebook)

| Setting | Mean OOF AUC | Std. dev. | Evaluations |
|---|---:|---:|---:|
| `C=0.01`, balanced | **0.615784** | 0.080582 | 36 |
| `C=0.03`, balanced | 0.609931 | 0.079175 | 36 |
| `C=1.0`, balanced (previous setting) | 0.600639 | 0.079823 | 36 |

Mean paired difference versus `C=1.0`: `C=0.01` = **+0.015145**; `C=0.03` = +0.009292. `C=0.01` has the strongest overall mean, but it is not best for every target (for example, ACL performs worse than with `C=1.0`).

### Interpretation / limitations

- The 36 target/seed evaluations are **not independent**: the same 58 labelled studies are reused across seeds.
- The labelled dataset is very small, and several hyperparameters have been compared, so the best observed result may be optimistic.
- OOF AUC is not the competition leaderboard score. This result supports using `C=0.01` as the current candidate, not claiming a proven rank improvement.
- The final model now uses `C=0.01` with `class_weight="balanced"`. This changes model behaviour. It does not alter feature extraction, slice sampling, target definitions, or submission validation.

Cells 13–14 are earlier experiments and can be skipped. Cell 15 is self-contained after setup cells 2–5. Re-run the full submission pipeline after changing the model and inspect `run_report.json`; the saved pipeline outputs from an earlier run are not evidence that the updated model has executed successfully.

### Final-pipeline run recorded (2026-10-16)

The uploaded notebook output shows the updated pipeline executed after the model configuration cell was run (`C=0.01`, execution count 9; pipeline cell execution count 11).

- **Status:** `OK`
- **Fully labelled training studies featurised:** 58/58 (18 s)
- **Target models fitted:** 12/12
- **Test studies predicted:** 3/3
- **Model predictions:** 3/3; prior-filled: 0
- **Device:** CUDA
- **Total elapsed time:** 18.9 s; extraction: 0.29 s/study
- **Validation:** `submission.csv` was written and re-validated.

This confirms the pipeline ran successfully in that notebook session and produced a validated file for the 3 test studies present in that run. It does **not** establish leaderboard performance, and the small test count should be checked against the official competition runtime/test split before submission.

### Clarification: why this run has 3 test studies

The saved `real_submission_audit.json` output for this notebook session reports `n_rows = 3` and `n_expected_test_rows = 3`. The pipeline takes prediction IDs directly from `DATA / "test.csv"` in `resolve_inputs()`; it does not hard-code a three-study limit. Therefore, the three rows describe the competition files visible in that run, not model capacity or model quality. Verify the test row count again in the official Kaggle submission run; this local/session output alone does not establish the size of the test set used for leaderboard scoring.

### Next modelling direction

The current representation averages the 9 slice embeddings into one 768-dimensional vector. This may discard useful slice-level information. Next experiment: compare mean pooling against a small, pre-specified set of alternative aggregations (for example, concatenated mean and standard deviation, or mean plus max) using identical out-of-fold splits. Keep all pooling choices evaluated only on the 58 gold-labelled studies, and only promote a change if the improvement is consistent enough to justify the extra feature dimensions and small-sample risk. Do not change the submission pipeline based on test predictions.

In [5]:

# OOF BASELINE EXPERIMENT — does not create or modify submission.csv
# Reuses the notebook's existing DINOv2 extraction and target definitions.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.base import clone
import pandas as pd
import numpy as np

print("Loading DINOv2 and extracting the 58 labelled studies...")
encode, device = load_encoder()
state = {"train_rungs": Counter(), "notes": []}
X, gold = build_training_set(encode, state)

print(f"Device: {device}")
print(f"Feature matrix: {X.shape}")
print(f"Label rows: {len(gold)}")

if len(X) != len(gold) or len(X) < 20:
    raise ValueError("Feature/label mismatch or too few usable studies. Stop and inspect the extraction logs.")

# Test the current model plus a small, pre-defined set of alternatives.
configs = [
    {"C": 0.01, "class_weight": "balanced"},
    {"C": 0.1,  "class_weight": "balanced"},
    {"C": 1.0,  "class_weight": "balanced"},  # current baseline
    {"C": 10.0, "class_weight": "balanced"},
    {"C": 0.1,  "class_weight": None},
    {"C": 1.0,  "class_weight": None},
    {"C": 10.0, "class_weight": None},
]

results = []
target_summary = []

for target in TARGETS:
    y = pd.to_numeric(gold[target], errors="coerce").to_numpy()
    valid = np.isfinite(y)
    Xt = X[valid]
    yt = y[valid].astype(int)

    counts = np.bincount(yt, minlength=2)
    n_minority = int(counts.min())

    target_summary.append({
        "target": target,
        "n": len(yt),
        "negatives": int(counts[0]),
        "positives": int(counts[1]),
    })

    # AUC cannot be evaluated if a target has only one class.
    if len(np.unique(yt)) != 2 or n_minority < 2:
        print(f"SKIP {target}: positives={counts[1]}, negatives={counts[0]}")
        continue

    n_splits = min(5, n_minority)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    # Identical folds for all configurations of this target.
    splits = list(cv.split(Xt, yt))

    for cfg in configs:
        oof = np.full(len(yt), np.nan)

        for train_idx, val_idx in splits:
            model = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    C=cfg["C"],
                    class_weight=cfg["class_weight"],
                    max_iter=2000,
                    random_state=SEED,
                ),
            )
            model.fit(Xt[train_idx], yt[train_idx])
            oof[val_idx] = model.predict_proba(Xt[val_idx])[:, 1]

        if not np.isfinite(oof).all():
            raise ValueError(f"Missing OOF predictions for {target}, config={cfg}")

        results.append({
            "target": target,
            "C": cfg["C"],
            "class_weight": str(cfg["class_weight"]),
            "folds": n_splits,
            "AUC": roc_auc_score(yt, oof),
            "log_loss": log_loss(yt, np.clip(oof, 1e-7, 1 - 1e-7), labels=[0, 1]),
        })

summary = pd.DataFrame(target_summary)
scores = pd.DataFrame(results)

print("\n=== Target class counts ===")
display(summary)

if scores.empty:
    print("No targets had enough examples of both classes to evaluate AUC.")
else:
    print("\n=== Per-target OOF results ===")
    display(scores.sort_values(["target", "AUC"], ascending=[True, False]))

    print("\n=== Macro average across evaluable targets ===")
    macro = (
        scores.groupby(["C", "class_weight"], as_index=False)
        .agg(mean_AUC=("AUC", "mean"), mean_log_loss=("log_loss", "mean"),
             targets_evaluated=("target", "nunique"))
        .sort_values("mean_AUC", ascending=False)
    )
    display(macro)

    print(
        "\nInterpretation: higher AUC is better; lower log loss is better. "
        "These are cross-validation diagnostics, not a confirmed competition-score improvement."
    )


Loading DINOv2 and extracting the 58 labelled studies...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Training features: 58/58 gold studies in 69s
Device: cuda
Feature matrix: (58, 768)
Label rows: 58

=== Target class counts ===


,target,n,negatives,positives
0,ACL,58,34,24
1,MCL,58,49,9
2,Medial Meniscus,58,32,26
3,Lateral Meniscus,58,35,23
4,Medial OA,58,43,15
5,Lateral OA,58,47,11
6,PF OA,58,37,21
7,Effusion,58,23,35
8,Synovitis,58,31,27
9,Baker's,58,46,12



=== Per-target OOF results ===


,target,C,class_weight,folds,AUC,log_loss
6,ACL,10.0,None,5,0.584559,1.745028
3,ACL,10.0,balanced,5,0.583333,1.745478
5,ACL,1.0,None,5,0.572304,1.340375
2,ACL,1.0,balanced,5,0.571078,1.334961
1,ACL,0.1,balanced,5,0.568627,0.976504
...,...,...,...,...,...,...
60,Synovitis,0.1,None,5,0.507766,1.184988
62,Synovitis,10.0,None,5,0.495818,2.465094
59,Synovitis,10.0,balanced,5,0.494624,2.471617
58,Synovitis,1.0,balanced,5,0.492234,1.787251



=== Macro average across evaluable targets ===


,C,class_weight,mean_AUC,mean_log_loss,targets_evaluated
0,0.01,balanced,0.627273,0.645695,12
2,0.10,balanced,0.615236,0.853205,12
1,0.10,None,0.614999,0.844310,12
3,1.00,None,0.610778,1.199295,12
4,1.00,balanced,0.610244,1.204745,12
6,10.00,balanced,0.609783,1.604024,12
5,10.00,None,0.609775,1.593444,12



Interpretation: higher AUC is better; lower log loss is better. These are cross-validation diagnostics, not a confirmed competition-score improvement.


In [19]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

# Compare only the current setting against the leading candidate.
configs = [
    {"name": "current_C1_balanced", "C": 1.0, "class_weight": "balanced"},
    {"name": "candidate_C001_balanced", "C": 0.01, "class_weight": "balanced"},
]

all_results = []

for target in TARGETS:
    y = pd.to_numeric(gold[target], errors="coerce").to_numpy()
    valid = np.isfinite(y)
    Xt, yt = X[valid], y[valid].astype(int)

    counts = np.bincount(yt, minlength=2)
    if len(np.unique(yt)) != 2 or counts.min() < 2:
        continue

    n_splits = min(5, int(counts.min()))

    for seed in [42, 123, 2026]:
        cv = StratifiedKFold(
            n_splits=n_splits, shuffle=True, random_state=seed
        )
        splits = list(cv.split(Xt, yt))

        for cfg in configs:
            oof = np.full(len(yt), np.nan)

            for train_idx, val_idx in splits:
                model = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        C=cfg["C"],
                        class_weight=cfg["class_weight"],
                        max_iter=2000,
                        random_state=42,
                    ),
                )
                model.fit(Xt[train_idx], yt[train_idx])
                oof[val_idx] = model.predict_proba(Xt[val_idx])[:, 1]

            all_results.append({
                "target": target,
                "seed": seed,
                "model": cfg["name"],
                "AUC": roc_auc_score(yt, oof),
            })

results_multi = pd.DataFrame(all_results)

wide = results_multi.pivot(
    index=["target", "seed"], columns="model", values="AUC"
).reset_index()

wide["candidate_minus_current"] = (
    wide["candidate_C001_balanced"] - wide["current_C1_balanced"]
)

print("Mean AUC by model across target/seed evaluations:")
display(
    results_multi.groupby("model")["AUC"]
    .agg(["mean", "std", "count"])
)

print("Mean candidate-minus-current difference by target:")
display(
    wide.groupby("target")["candidate_minus_current"]
    .agg(["mean", "std", "count"])
    .sort_values("mean")
)

print("Overall paired difference:")
print(wide["candidate_minus_current"].mean())


Mean AUC by model across target/seed evaluations:


,mean,std,count
model,,,
candidate_C001_balanced,0.615784,0.080582,36
current_C1_balanced,0.600639,0.079823,36


Mean candidate-minus-current difference by target:


,mean,std,count
target,,,
ACL,-0.045752,0.033367,3
MCL,-0.005291,0.032175,3
Contusion,-0.004498,0.004123,3
Effusion,-0.003727,0.020450,3
Medial OA,0.002067,0.003227,3
Lateral OA,0.004513,0.006218,3
Baker's,0.016304,0.011019,3
Lateral Meniscus,0.018634,0.006573,3
PF OA,0.043329,0.021625,3


Overall paired difference:
0.015145195578898488


In [ ]:
# THREE-WAY OOF COMPARISON — self-contained after setup cells 2–5.
# This cell does NOT change the final submission model or write submission.csv.
# It rebuilds X/gold itself, so it does not depend on running experiment cells 13 or 14.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import gc

print("Extracting features for the fully labelled training studies...")
encode, device = load_encoder()
state = {"train_rungs": Counter(), "notes": []}
X, gold = build_training_set(encode, state)
del encode
gc.collect()

print(f"Device: {device} | X shape: {X.shape} | labelled rows: {len(gold)}")
if len(X) != len(gold) or len(X) < 20:
    raise ValueError("Feature/label mismatch or too few usable studies; inspect the extraction logs before continuing.")

configs = [
    {"name": "C=1.0", "C": 1.0},
    {"name": "C=0.01", "C": 0.01},
    {"name": "C=0.03", "C": 0.03},
]
seeds = [42, 123, 2026]
all_results_c = []

for target in TARGETS:
    y_raw = pd.to_numeric(gold[target], errors="coerce").to_numpy()
    valid = np.isfinite(y_raw)
    Xt, yt = X[valid], y_raw[valid].astype(int)
    counts = np.bincount(yt, minlength=2)

    if len(np.unique(yt)) != 2 or counts.min() < 2:
        print(f"SKIP {target}: positives={counts[1]}, negatives={counts[0]}")
        continue

    n_splits = min(5, int(counts.min()))
    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(cv.split(Xt, yt))  # identical folds for all three C values

        for cfg in configs:
            oof = np.full(len(yt), np.nan)
            for train_idx, val_idx in splits:
                model = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        C=cfg["C"],
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=42,
                    ),
                )
                model.fit(Xt[train_idx], yt[train_idx])
                oof[val_idx] = model.predict_proba(Xt[val_idx])[:, 1]

            if not np.isfinite(oof).all():
                raise ValueError(f"Non-finite OOF predictions for {target}, {cfg['name']}, seed={seed}")

            all_results_c.append({
                "target": target,
                "seed": seed,
                "model": cfg["name"],
                "AUC": roc_auc_score(yt, oof),
                "n": len(yt),
                "positives": int(counts[1]),
                "negatives": int(counts[0]),
            })

results_c = pd.DataFrame(all_results_c)
if results_c.empty:
    raise ValueError("No targets produced valid OOF AUC results.")

print("Mean AUC across target/seed evaluations (descriptive, not independent samples):")
display(results_c.groupby("model")["AUC"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False))

wide_c = results_c.pivot(index=["target", "seed"], columns="model", values="AUC").reset_index()
print("\nPaired mean AUC differences versus C=1.0:")
for name in ["C=0.01", "C=0.03"]:
    diff = wide_c[name] - wide_c["C=1.0"]
    print(f"{name} minus C=1.0: {diff.mean():+.6f}")

print("\nPer-target mean AUC:")
display(results_c.groupby(["target", "model"])["AUC"].mean().unstack("model"))

print("\nPositive/negative counts by target:")
display(results_c[["target", "n", "positives", "negatives"]].drop_duplicates().sort_values("target"))


### Experiment: preserve slice-level variation

The current model averages 9 slice embeddings into 768 features. This experiment compares three pre-defined representations on the same labelled studies and identical OOF folds:

- `mean`: current 768-dimensional representation.
- `mean + std`: mean embedding plus per-feature variation across slices (1536 dimensions).
- `mean + max`: mean embedding plus per-feature maximum across slices (1536 dimensions).

It keeps the current `C=0.01`, balanced logistic regression fixed so we isolate the pooling choice. It reports per-target and macro OOF AUC over three seeds. This is an experiment only; it does not change the submission model or write `submission.csv`. Run after setup cells 2–5. Feature extraction takes about 1–2 minutes.

In [ ]:
# SLICE-POOLING OOF EXPERIMENT — does not change the submission pipeline.
# Rebuilds features for gold studies, preserving the 9 slice embeddings per study.
# Same target/seed folds are used for each pooling method.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import gc

print("Extracting per-slice DINOv2 features for labelled studies...")
encode, device = load_encoder()
labels = pd.read_csv(DATA / "train.csv")
series = pd.read_csv(DATA / "train_series.csv")
gold_all = labels.dropna(subset=TARGETS).sort_values("StudyInstanceUID").reset_index(drop=True)

feature_rows = []
kept_ids = []
pooling_state = {"train_rungs": Counter(), "notes": []}
for _, row in gold_all.iterrows():
    sid = str(row["StudyInstanceUID"])
    try:
        f, info = extract_study(encode, series, DATA / "train_series", sid)
        f = np.asarray(f, dtype=np.float32)
        if f.ndim != 2 or f.shape[1] != 768 or f.shape[0] == 0:
            raise ValueError(f"Unexpected per-slice feature shape: {f.shape}")
        feature_rows.append({
            "mean": f.mean(axis=0),
            "mean+std": np.concatenate([f.mean(axis=0), f.std(axis=0)]),
            "mean+max": np.concatenate([f.mean(axis=0), f.max(axis=0)]),
        })
        kept_ids.append(sid)
        pooling_state["train_rungs"][info["rung"]] += 1
    except Exception as e:
        log("pooling_experiment_train_failed", study=sid, error=repr(e)[:200])

del encode
gc.collect()
gold_pool = gold_all[gold_all["StudyInstanceUID"].astype(str).isin(kept_ids)].reset_index(drop=True)
if len(feature_rows) != len(gold_pool) or len(gold_pool) < 20:
    raise ValueError(f"Feature/label mismatch or too few usable studies: {len(feature_rows)} features, {len(gold_pool)} labels")
print(f"Device: {device} | usable studies: {len(gold_pool)}/{len(gold_all)}")
print("Feature dimensions:", {k: v.shape for k, v in feature_rows[0].items()})

pool_results = []
seeds_pool = [42, 123, 2026]
for target in TARGETS:
    y_raw = pd.to_numeric(gold_pool[target], errors="coerce").to_numpy()
    valid = np.isfinite(y_raw)
    yt = y_raw[valid].astype(int)
    counts = np.bincount(yt, minlength=2)
    if len(np.unique(yt)) != 2 or counts.min() < 2:
        print(f"SKIP {target}: positives={counts[1]}, negatives={counts[0]}")
        continue

    n_splits = min(5, int(counts.min()))
    for seed in seeds_pool:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        # Build each target's exact same folds for all pooling representations.
        splits = list(cv.split(np.zeros(len(yt)), yt))
        for pooling_name in ["mean", "mean+std", "mean+max"]:
            X_pool = np.stack([row[pooling_name] for row in feature_rows])[valid]
            oof = np.full(len(yt), np.nan)
            for train_idx, val_idx in splits:
                model = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(C=0.01, class_weight="balanced",
                                       max_iter=2000, random_state=42),
                )
                model.fit(X_pool[train_idx], yt[train_idx])
                oof[val_idx] = model.predict_proba(X_pool[val_idx])[:, 1]
            if not np.isfinite(oof).all():
                raise ValueError(f"Non-finite OOF predictions: {target}, {pooling_name}, seed={seed}")
            pool_results.append({
                "target": target, "seed": seed, "pooling": pooling_name,
                "AUC": roc_auc_score(yt, oof), "n": len(yt),
                "positives": int(counts[1]), "negatives": int(counts[0]),
            })

pool_scores = pd.DataFrame(pool_results)
if pool_scores.empty:
    raise ValueError("No valid OOF results produced.")
print("\nMacro OOF AUC across target/seed evaluations (descriptive, not independent samples):")
display(pool_scores.groupby("pooling")["AUC"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False))
print("\nPer-target mean OOF AUC:")
display(pool_scores.groupby(["target", "pooling"])["AUC"].mean().unstack("pooling"))
print("\nThis experiment does not modify fit_models() or submission.csv.")


### Experiment: slice-range selection

The current sampler uses 9 evenly spaced slices across the full series. This experiment compares three pre-specified ranges: the full series `(0.0, 1.0)`, a mildly central range `(0.1, 0.9)`, and a more central range `(0.2, 0.8)`. It keeps DINOv2, mean pooling, `C=0.01`, labels, and target/seed CV folds fixed.

Feature extraction is repeated for each range. Only studies successfully extracted under all three settings are evaluated, so the comparisons use the same studies. The original global `SLICE_RANGE` is restored even if extraction raises an exception. This is an experiment only: it does not modify the final model or write `submission.csv`.

Run after setup cells 2–5. Expect a few minutes of GPU work.

In [ ]:
# SLICE-RANGE OOF EXPERIMENT — does not change the submission pipeline.
# Compares full-series vs central-slice sampling with fixed model/folds.
# Does not write submission.csv.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import gc

slice_range_candidates = [
    ("full_0.0_1.0", (0.0, 1.0)),
    ("central_0.1_0.9", (0.1, 0.9)),
    ("central_0.2_0.8", (0.2, 0.8)),
]
original_slice_range = SLICE_RANGE
slice_features = {}
slice_info = {}
slice_failures = []
slice_device = None

labels_slice = pd.read_csv(DATA / "train.csv")
series_slice = pd.read_csv(DATA / "train_series.csv")
gold_slice_all = (
    labels_slice.dropna(subset=TARGETS)
    .sort_values("StudyInstanceUID")
    .reset_index(drop=True)
)
gold_ids_all = gold_slice_all["StudyInstanceUID"].astype(str).tolist()

print("Extracting features for each pre-specified slice range...")
encode_slice, slice_device = load_encoder()

try:
    for range_name, candidate_range in slice_range_candidates:
        SLICE_RANGE = candidate_range
        rows_by_id = {}
        infos_by_id = {}

        print(f"\nSlice range: {range_name} {candidate_range}")
        for i, sid in enumerate(gold_ids_all, start=1):
            try:
                f, info = extract_study(
                    encode_slice, series_slice, DATA / "train_series", sid
                )
                f = np.asarray(f, dtype=np.float32)
                if f.shape != (N_SLICES, 768) or not np.isfinite(f).all():
                    raise ValueError(f"Unexpected feature shape/content: {f.shape}")
                rows_by_id[sid] = f.mean(axis=0)
                infos_by_id[sid] = info
            except Exception as e:
                slice_failures.append({
                    "range": range_name,
                    "study": sid,
                    "error": repr(e)[:300],
                })

            if i % 10 == 0 or i == len(gold_ids_all):
                print(
                    f"Processed {i}/{len(gold_ids_all)}; "
                    f"successful={len(rows_by_id)}"
                )

        slice_features[range_name] = rows_by_id
        slice_info[range_name] = infos_by_id

finally:
    SLICE_RANGE = original_slice_range
    del encode_slice
    gc.collect()

common_ids = [
    sid for sid in gold_ids_all
    if all(sid in slice_features[name] for name, _ in slice_range_candidates)
]
gold_slice = gold_slice_all[
    gold_slice_all["StudyInstanceUID"].astype(str).isin(common_ids)
].copy().reset_index(drop=True)

print(f"\nDevice: {slice_device}")
print(f"Common usable studies: {len(common_ids)}/{len(gold_ids_all)}")
print(f"Extraction failures across ranges: {len(slice_failures)}")
print(f"Restored SLICE_RANGE: {SLICE_RANGE}")

if slice_failures:
    print("First 10 extraction failures:")
    display(pd.DataFrame(slice_failures).head(10))

if len(common_ids) < 20:
    raise ValueError(
        f"Only {len(common_ids)} common studies across slice ranges; "
        "too few for a meaningful comparison."
    )

X_by_range = {
    name: np.stack([slice_features[name][sid] for sid in common_ids]).astype(np.float32)
    for name, _ in slice_range_candidates
}
print("Feature matrix shapes:", {
    name: X.shape for name, X in X_by_range.items()
})

slice_oof_results = []
slice_seeds = [42, 123, 2026]

for target in TARGETS:
    y_raw = pd.to_numeric(gold_slice[target], errors="coerce").to_numpy()
    valid = np.isfinite(y_raw)
    yt = y_raw[valid].astype(int)
    counts = np.bincount(yt, minlength=2)

    if len(np.unique(yt)) != 2 or counts.min() < 2:
        print(f"SKIP {target}: positives={counts[1]}, negatives={counts[0]}")
        continue

    n_splits = min(5, int(counts.min()))
    for seed in slice_seeds:
        cv = StratifiedKFold(
            n_splits=n_splits, shuffle=True, random_state=seed
        )
        splits = list(cv.split(np.zeros(len(yt)), yt))

        for range_name, _ in slice_range_candidates:
            X_target = X_by_range[range_name][valid]
            oof = np.full(len(yt), np.nan, dtype=np.float64)

            for train_idx, val_idx in splits:
                model = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        C=0.01,
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=42,
                    ),
                )
                model.fit(X_target[train_idx], yt[train_idx])
                oof[val_idx] = model.predict_proba(X_target[val_idx])[:, 1]

            if not np.isfinite(oof).all():
                raise ValueError(
                    f"Non-finite OOF predictions: {target}, {range_name}, seed={seed}"
                )

            slice_oof_results.append({
                "target": target,
                "seed": seed,
                "slice_range": range_name,
                "AUC": roc_auc_score(yt, oof),
                "n": len(yt),
                "positives": int(counts[1]),
                "negatives": int(counts[0]),
            })

slice_scores = pd.DataFrame(slice_oof_results)
if slice_scores.empty:
    raise ValueError("No valid slice-range OOF results were produced.")

print(
    "\nMacro OOF AUC by slice range "
    "(descriptive; target/seed evaluations are not independent):"
)
display(
    slice_scores.groupby("slice_range")["AUC"]
    .agg(["mean", "std", "count"])
    .sort_values("mean", ascending=False)
)

print("\nPer-target mean OOF AUC:")
display(
    slice_scores.groupby(["target", "slice_range"])["AUC"]
    .mean()
    .unstack("slice_range")
)

print("\nThis experiment does not modify fit_models() or submission.csv.")

### Slice-range experiment — observed findings (2026-10-16)

**Status:** Completed successfully in the user's Kaggle run. These results were supplied from the executed cell output; they have not been independently rerun by this repository update.

- All 58/58 fully labelled studies were extracted successfully for each of the three pre-specified ranges; zero extraction failures.
- Feature matrices were all shaped (58, 768).
- The original `SLICE_RANGE=(0.0, 1.0)` was restored after feature extraction.
- Model configuration stayed fixed: mean pooling, DINOv2-Base embeddings, `C=0.01`, balanced logistic regression, seeds 42/123/2026.

| Slice range | Mean OOF AUC | Std | Evaluations |
|---|---:|---:|---:|
| Central 80% (0.1, 0.9) | 0.625780 | 0.095379 | 36 |
| Full range (0.0, 1.0) | 0.615784 | 0.080582 | 36 |
| Central 60% (0.2, 0.8) | 0.608889 | 0.111737 | 36 |

Central 80% exceeds the full range by +0.009996 mean AUC in this experiment. Target-level results are mixed: central 80% improved some labels (including Baker's, contusion and effusion) but reduced others (including MCL and synovitis). With only 58 labelled studies, the repeated target/seed evaluations are not independent and this is exploratory evidence, not proof of generalisation.

**Decision:** Keep the production pipeline unchanged at `SLICE_RANGE=(0.0, 1.0)`. Treat `(0.1, 0.9)` as the leading slice-range candidate and run the follow-up seed robustness experiment below using the already extracted `X_by_range` features. Do not select the final setting based on these results alone.

### Follow-up: slice-range seed robustness

Run the code cell below only after the slice-range OOF experiment has completed in the same Kaggle session. It reuses extracted features, so it should be substantially cheaper than re-extracting DINOv2 embeddings. It does not modify the final model or submission file.

In [ ]:
# SLICE-RANGE ROBUSTNESS CHECK — uses features from the preceding slice-range cell.
# Adds five pre-specified split seeds; does not re-extract images or change the submission pipeline.
# Run only after the SLICE-RANGE OOF EXPERIMENT cell has completed in this session.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

required = ["gold_slice", "X_by_range", "slice_range_candidates", "TARGETS"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables {missing}. Run the preceding SLICE-RANGE OOF EXPERIMENT cell first."
    )

# These seeds were not used in the initial three-seed slice-range experiment.
robustness_seeds = [7, 21, 2025, 31415, 8675309]
robustness_results = []

for target in TARGETS:
    y_raw = pd.to_numeric(gold_slice[target], errors="coerce").to_numpy()
    valid = np.isfinite(y_raw)
    yt = y_raw[valid].astype(int)
    counts = np.bincount(yt, minlength=2)

    if len(np.unique(yt)) != 2 or counts.min() < 2:
        print(f"SKIP {target}: positives={counts[1]}, negatives={counts[0]}")
        continue

    n_splits = min(5, int(counts.min()))
    for seed in robustness_seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(cv.split(np.zeros(len(yt)), yt))

        for range_name, _ in slice_range_candidates:
            X_target = X_by_range[range_name][valid]
            oof = np.full(len(yt), np.nan, dtype=np.float64)

            for train_idx, val_idx in splits:
                model = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        C=0.01, class_weight="balanced",
                        max_iter=2000, random_state=42
                    ),
                )
                model.fit(X_target[train_idx], yt[train_idx])
                oof[val_idx] = model.predict_proba(X_target[val_idx])[:, 1]

            if not np.isfinite(oof).all():
                raise ValueError(
                    f"Non-finite OOF predictions: {target}, {range_name}, seed={seed}"
                )

            robustness_results.append({
                "target": target,
                "seed": seed,
                "slice_range": range_name,
                "AUC": roc_auc_score(yt, oof),
            })

robustness_scores = pd.DataFrame(robustness_results)
if robustness_scores.empty:
    raise ValueError("No valid robustness results produced.")

print("Additional-seed mean OOF AUC (descriptive; not independent observations):")
display(
    robustness_scores.groupby("slice_range")["AUC"]
    .agg(["mean", "std", "count"])
    .sort_values("mean", ascending=False)
)

wide_robustness = robustness_scores.pivot(
    index=["target", "seed"], columns="slice_range", values="AUC"
).reset_index()
wide_robustness["central80_minus_full"] = (
    wide_robustness["central_0.1_0.9"] - wide_robustness["full_0.0_1.0"]
)
wide_robustness["central60_minus_full"] = (
    wide_robustness["central_0.2_0.8"] - wide_robustness["full_0.0_1.0"]
)

print("\nPaired differences by target across the additional seeds:")
display(
    wide_robustness.groupby("target")[
        ["central80_minus_full", "central60_minus_full"]
    ].agg(["mean", "std", "count"])
)

print("\nMean paired differences across target/seed rows:")
print("Central 80% minus full:", round(wide_robustness["central80_minus_full"].mean(), 6))
print("Central 60% minus full:", round(wide_robustness["central60_minus_full"].mean(), 6))

print(
    "\nInterpret cautiously: this checks sensitivity to split seeds on the same 58 studies; "
    "it is not an independent test set and cannot establish leaderboard improvement."
)

## 5. EXP-007 parser audit — separate research workflow (skip for submission)

The remaining cells clone and inspect a different research branch for parser diagnostics and weak-label audits. **They are not part of the knee submission pipeline. Do not run them when generating `submission.csv`.** They depend on separate repository files and may change directories or run tests unrelated to inference.

In [10]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/rsna-knee-m1-exp007")
branch = "research/exp007-gold-agreement-audit"
url = "https://github.com/vixintry/rsna-knee-m1.git"

if not (repo / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", branch, "--single-branch", url, str(repo)],
        check=True
    )
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", branch], check=True)
    subprocess.run(
        ["git", "-C", str(repo), "checkout", "-B", branch, f"origin/{branch}"],
        check=True
    )

script = repo / "scripts" / "audit_exp007_gold_agreement.py"
assert script.exists(), f"Audit script missing: {script}"

subprocess.run([sys.executable, str(script)], check=True)

Cloning into '/kaggle/working/rsna-knee-m1-exp007'...


Gold studies: 58

Per-target agreement and coverage:
          target  covered  coverage  tp  fp  tn  fn  precision_on_covered  recall_on_covered  unknown_status  uncertain_status  conflict_status
             ACL       27  0.465517  13   5   9   0              0.722222           1.000000              31                 0                0
             MCL       20  0.344828   5   4  11   0              0.555556           1.000000              37                 1                0
 Medial Meniscus       18  0.310345   9   3   6   0              0.750000           1.000000              39                 1                0
Lateral Meniscus       18  0.310345   8   3   7   0              0.727273           1.000000              39                 1                0
       Medial OA        1  0.017241   1   0   0   0              1.000000           1.000000              57                 0                0
      Lateral OA        0  0.000000   0   0   0   0                   NaN          

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/rsna-knee-m1-exp007/scripts/audit_exp007_gold_agreement.py'], returncode=0)

In [11]:
%cd /kaggle/working/rsna-knee-m1-exp007
!git fetch origin research/exp007-parser-v2-diagnostics
!git checkout -B research/exp007-parser-v2-diagnostics FETCH_HEAD

/kaggle/working/rsna-knee-m1-exp007
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 37 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (37/37), 13.42 KiB | 361.00 KiB/s, done.
From https://github.com/vixintry/rsna-knee-m1
 * branch            research/exp007-parser-v2-diagnostics -> FETCH_HEAD
Switched to a new branch 'research/exp007-parser-v2-diagnostics'


In [12]:
!python -m unittest tests.test_derive_report_weak_labels_v2 -v

test_derive_report_weak_labels_v2 (unittest.loader._FailedTest.test_derive_report_weak_labels_v2) ... ERROR

ERROR: test_derive_report_weak_labels_v2 (unittest.loader._FailedTest.test_derive_report_weak_labels_v2)
----------------------------------------------------------------------
ImportError: Failed to import test module: test_derive_report_weak_labels_v2
Traceback (most recent call last):
  File "/usr/lib/python3.13/unittest/loader.py", line 141, in loadTestsFromName
    module = __import__(module_name)
ModuleNotFoundError: No module named 'tests.test_derive_report_weak_labels_v2'


----------------------------------------------------------------------
Ran 1 test in 0.000s

FAILED (errors=1)


In [13]:
%cd /kaggle/working/rsna-knee-m1-exp007
!python -m unittest discover -s tests -p "test_derive_report_weak_labels_v2.py" -v

/kaggle/working/rsna-knee-m1-exp007
test_complete_mcl_rupture_is_not_overridden_by_missing_acute_word (test_derive_report_weak_labels_v2.ParserV2GuardTests.test_complete_mcl_rupture_is_not_overridden_by_missing_acute_word) ... ok
test_explicit_acl_no_tear_is_negative (test_derive_report_weak_labels_v2.ParserV2GuardTests.test_explicit_acl_no_tear_is_negative) ... ok
test_explicit_intact_mcl_stays_negative_with_other_injury (test_derive_report_weak_labels_v2.ParserV2GuardTests.test_explicit_intact_mcl_stays_negative_with_other_injury) ... ok
test_grade_one_mcl_sprain_is_negative (test_derive_report_weak_labels_v2.ParserV2GuardTests.test_grade_one_mcl_sprain_is_negative) ... ok
test_high_grade_acl_tear_not_overridden_by_low_grade_mcl (test_derive_report_weak_labels_v2.ParserV2GuardTests.test_high_grade_acl_tear_not_overridden_by_low_grade_mcl) ... ok
test_high_grade_acl_tear_remains_positive (test_derive_report_weak_labels_v2.ParserV2GuardTests.test_high_grade_acl_tear_remains_positive) .

In [14]:
from pathlib import Path
import pandas as pd

root = Path("/kaggle/working/exp007_parser_v2_comparison")
detail_path = root / "parser_v2_gold_detail.csv"

print("Directory exists:", root.exists())
print("Detail file exists:", detail_path.exists())

if detail_path.exists():
    df = pd.read_csv(detail_path)
    print("Columns:", df.columns.tolist())
    display(df.head(5))

Directory exists: False
Detail file exists: False


In [15]:
%cd /kaggle/working/rsna-knee-m1-exp007

!git status --short --branch
!ls scripts
!ls reports

/kaggle/working/rsna-knee-m1-exp007
## research/exp007-parser-v2-diagnostics
audit_exp007_parser_v2.py  derive_report_weak_labels.py     __pycache__
create_oof_folds.py	   derive_report_weak_labels_v2.py
2026-10-10-dicom-spatial-ordering-audit.md
2026-10-10-first-model-results.md
2026-10-10-pretrained-model-research.md
2026-10-11-exp007-weak-label-plan.md
2026-10-13-exp007-provenance-reconciliation.md
2026-10-14-exp007-parser-v2-diagnostic.md
2026-10-15-exp007-parser-v2-case-review.md


In [16]:
%cd /kaggle/working/rsna-knee-m1-exp007
!PYTHONPATH=. python scripts/audit_exp007_parser_v2.py --help

/kaggle/working/rsna-knee-m1-exp007
usage: audit_exp007_parser_v2.py [-h] [--input-csv INPUT_CSV]
                                 [--output-dir OUTPUT_DIR]
                                 [--expected-gold-studies EXPECTED_GOLD_STUDIES]

options:
  -h, --help            show this help message and exit
  --input-csv INPUT_CSV
  --output-dir OUTPUT_DIR
  --expected-gold-studies EXPECTED_GOLD_STUDIES


In [17]:
#PERM

%cd /kaggle/working/rsna-knee-m1-exp007

!PYTHONPATH=. python scripts/audit_exp007_parser_v2.py \
    --output-dir /kaggle/working/exp007_parser_v2_comparison \
    --expected-gold-studies 58

/kaggle/working/rsna-knee-m1-exp007
          target  version  covered  coverage  tp  fp  tn  fn  precision   recall  status_unknown  status_positive  status_negative  status_uncertain  status_conflict
             ACL baseline       27  0.465517  13   5   9   0   0.722222 1.000000              31             18.0              9.0               NaN              NaN
             ACL       v2       29  0.500000  13   2  14   0   0.866667 1.000000              29             15.0             14.0               NaN              NaN
             MCL baseline       20  0.344828   5   4  11   0   0.555556 1.000000              37              9.0             11.0               1.0              NaN
             MCL       v2       25  0.431034   3   0  19   3   1.000000 0.500000              33              3.0             22.0               NaN              NaN
 Medial Meniscus baseline       18  0.310345   9   3   6   0   0.750000 1.000000              39             12.0              6.0    

## EXP-007 Parser V2 Audit — Final Findings

### Scope
Compared the baseline report parser and experimental parser V2 on 58 fully labelled gold studies. Gold labels were used for evaluation only and were not used for weak-label training.

### Validation
- Parser V2 unit tests passed: 16/16.
- The audit compared 12 targets and found 26 changed predictions.
- V2 improved precision for ACL, MCL, lateral meniscus, effusion and Baker's cyst on this audit, but reduced MCL and effusion recall.
- Metrics are conditional on parser coverage; unknown predictions are not treated as negative.

### Investigation decisions
1. **MCL:** The suspected bug was not reproduced. The inspected report contains a complete MCL rupture, and V2 correctly predicts positive.
2. **Effusion:** Small/trace effusions remain negative under the official moderate/large target criterion. Gold-positive disagreements are not sufficient justification to relax this threshold.
3. **Contusion:** Three disagreements were reviewed. Two gold-negative reports explicitly mention bone contusions; one gold-positive report explicitly denies bone contusion and describes a muscle contusion. No parser change is justified by these cases.
4. **Training integrity:** Gold-labelled studies remain evaluation-only and must not be added to weak-label training.

### Conclusion
No additional parser changes are justified by the reviewed disagreements. Preserve the current V2 rules, document the audit, and move on to the next experiment. This is a rule-based parser audit, not clinical validation.

## Reference notes

- To change the model, change `build_training_set` / `fit_models` only; `extract_study` is used identically for training and test, which keeps train/test features consistent.
- If you change `SLICE_RANGE`, `N_SLICES` or preprocessing, rerun the five-fold evaluation first; previous experiment results apply only to their recorded settings.
- `run_report.json` lists fallbacks (series, slice, study). A high count of `series_failed` or `study_failed` is the first thing to investigate.
- Current model candidate: `LogisticRegression(C=0.01, class_weight="balanced")`, based on the three-way OOF comparison recorded in section 4. This is provisional because only 58 fully labelled studies are available; do not describe it as a verified leaderboard improvement.